## 1. Visión del problema y selección de variables

El alumnado deberá:

- Definir un problema de Machine Learning realista (clasificación, regresión, clustering, etc.).
- Describir el contexto del problema y su utilidad.
- Identificar las variables relevantes que influyen en el problema.
- Justificar la selección de dichas variables.
- Definir claramente:
  - Variable objetivo (target).
  - Variables independientes (features).
  - Tipo de problema (supervisado / no supervisado).


## Contexto del Problema y su utilidad real
El avance de las tecnologías digitales y el crecimiento de las redes sociales han incrementado considerablemente la generación y difusión de fake news en internet. La rapidez con la que este tipo de información se comparte provoca que muchas noticias falsas lleguen a millones de personas antes de poder ser verificadas, generando problemas de desinformación, manipulación social y pérdida de confianza en los medios de comunicación. Debido al gran volumen de información que circula diariamente, la detección manual de este contenido resulta cada vez más complicada.

En este contexto, las tecnologías de Inteligencia Artificial y Machine Learning permiten desarrollar sistemas capaces de analizar automáticamente noticias y detectar patrones asociados a contenido potencialmente falso. Este proyecto tiene como objetivo desarrollar un sistema inteligente de detección de fake news utilizando técnicas de procesamiento de lenguaje natural (NLP) y modelos de clasificación supervisada. El funcionamiento del sistema será similar al utilizado por los filtros de detección de spam en correos electrónicos, donde el modelo aprende a identificar características comunes presentes en noticias falsas y reales para clasificarlas automáticamente.

La idea es un sistema de detección automática de fake news puede resultar útil en múltiples contextos reales:

* Plataformas digitales y redes sociales
* Medios de comunicación
* Organismos públicos
* Sistemas de moderación automática
* Herramientas de verificación de información

La automatización de este proceso permitiría detectar contenido sospechoso de forma rápida y escalable, reduciendo el impacto de la desinformación y facilitando tareas de análisis y supervisión de contenido digital.

## Identificación y Justificación de las Fuentes de Datos
Para el desarrollo de este modelo de detección de noticias falsas, se han seleccionado diversas fuentes de datos que permiten equilibrar el entrenamiento masivo con pruebas en entornos reales.

**1. Datasets de Entrenamiento (Cuerpo del texto completo)**
Estas fuentes constituyen el núcleo del aprendizaje del modelo debido a su volumen y balanceo:

>Kaggle (Fake and Real News Dataset): Compuesto por 23,502 noticias falsas y 21,417 reales. Proporciona una base equilibrada para el entrenamiento inicial.
URL: https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset

>Hugging Face (Fake News Detection Dataset): Complementa al anterior con una estructura similar de clasificación binaria (0 = real, 1 = falso).
URL: https://huggingface.co/datasets/ErfanMoosaviMonazzah/fake-news-detection-dataset-English

**2. Validación y Pruebas (Datos Dinámicos y Específicos)**
Fuentes utilizadas para testear la capacidad de generalización del modelo:

>NewsAPI: Se empleará para extraer noticias actuales mediante su API. Al no poseer un filtro de veracidad, se utilizará para testear el rendimiento del modelo frente a medios desconocidos o no verificados en tiempo real.
URL: https://newsapi.org/docs/get-started#search


**3. Datos de Referencia y Metadatos**
>FakeNewsNet: Repositorio que proporciona los enlaces originales (URLs) y metadatos sociales de noticias ya verificadas, útil para análisis de procedencia.
URL: https://github.com/KaiDMML/FakeNewsNet/tree/master/dataset

## Variables


> **Kaggle (Fake and Real News Dataset)** En este dataset la variable objetivo (target) será label, la cual indicará si una noticia es real o fake. Las variables independientes (features) principales utilizadas por el modelo serán title, correspondiente al titular de la noticia; text, que contiene el contenido completo; subject, asociado a la temática o categoría; date, que representa la fecha de publicación. Estas variables contienen información textual y contextual relevante que permitirá al modelo identificar patrones característicos presentes en noticias falsas y reales.

>**Hugging Face (Fake News Detection Dataset)** Este dataset presenta una estructura similar al dataset utilizado desde Kaggle, ya que también contiene noticias etiquetadas como reales o falsas mediante la variable label. La principal diferencia es que, en este caso, toda la información se encuentra integrada en un único archivo CSV, mientras que en el dataset de Kaggle las noticias reales y falsas se distribuyen en archivos separados. Exceptuando la columna label, el resto de variables utilizadas mantienen una estructura muy similar, incluyendo campos relacionados con el titular, contenido y metadatos de las noticias.

>**NewsAPI**
La API NewsAPI proporciona noticias en formato JSON obtenidas en tiempo real desde diferentes medios digitales. A diferencia de los datasets anteriores, esta fuente no incluye una variable objetivo (label) que indique si una noticia es real o falsa, por lo que será utilizada principalmente para pruebas y validación del sistema una vez entrenado el modelo. Las principales variables independientes (features) obtenidas desde esta API serán source.name (medio de comunicación), author (autor de la noticia), title (titular), description (descripción breve), content (contenido parcial de la noticia) y publishedAt (fecha de publicación). Estas variables aportan información textual y contextual relevante para evaluar el comportamiento del modelo sobre noticias reales obtenidas en tiempo real.

>**FakeNewsNet**
Este dataset contiene noticias almacenadas en formato JSON e incluye tanto información textual como metadatos asociados a cada publicación. A diferencia de otras fuentes, incorpora una estructura más compleja y semiestructurada, lo que permite trabajar con datos más cercanos a un entorno real de producción. En este caso, la variable objetivo (target) puede obtenerse a partir del campo trust.categories, donde se identifica si la noticia pertenece a la categoría fake_news. Las principales variables independientes (features) utilizadas serán title (titular de la noticia), text (contenido textual), author (autor), published (fecha de publicación), language (idioma), sentiment (sentimiento asociado al texto), categories (categorías temáticas), topics (temáticas relacionadas), así como variables relacionadas con la fuente de la noticia y métricas sociales presentes en la estructura thread.social. Estas variables aportan información contextual y semántica relevante para la detección automática de fake news.

# Justificación de la Arquitectura de Datos

El diseño de la arquitectura de ingesta se ha tomado en base a la naturaleza de cada fuente de datos, eligiendo en cada caso la tecnología más adecuada para el tipo de información que se maneja.


## 1. Datasets estáticos: Kaggle y Hugging Face → Amazon S3

Los datasets de Kaggle (ISOT Fake News Dataset) y Hugging Face (Fake News Detection Dataset English) se almacenan directamente en Amazon S3 por una razón fundamental: ambos se distribuyen en formato tabular estructurado (CSV y Parquet respectivamente), lo que los convierte en candidatos naturales para un almacenamiento de objetos en crudo. S3 es la opción más sencilla y económica para este caso, ya que permite almacenar archivos de gran volumen sin necesidad de definir un esquema previo, con alta durabilidad y accesibilidad directa desde cualquier servicio de AWS, en particular desde AWS Glue en la fase de transformación. Al tratarse de datos estáticos que no cambian en el tiempo, no tiene sentido introducir ninguna capa de procesamiento en tiempo real: se suben una vez y quedan disponibles para el ETL.


## 2. Noticias dinámicas: NewsAPI → Kafka → MongoDB Atlas

Las noticias obtenidas a través de NewsAPI tienen una naturaleza completamente distinta: son datos dinámicos, se consumen en tiempo real y llegan en formato JSON con una estructura que puede variar entre artículos (campos opcionales, objetos anidados, longitudes variables). Por este motivo se han elegido dos tecnologías complementarias para esta fuente.

En primer lugar, Kafka actúa como capa de mensajería entre la descarga de los artículos y su persistencia. Esto permite cumplir con el requisito de arquitectura streaming del proyecto, desacoplando el productor (el script que llama a NewsAPI) del consumidor (el proceso que escribe en base de datos), lo que en un entorno de producción permitiría escalar ambos de forma independiente y garantizar que ningún artículo se pierde en caso de fallos puntuales.

En segundo lugar, MongoDB Atlas recibe los artículos como documentos JSON sin forzar un esquema rígido. Esto es especialmente adecuado para datos de API, donde la estructura puede cambiar sin previo aviso y donde imponer un esquema relacional en la fase de ingesta cruda introduciría una fragilidad innecesaria en el pipeline.

## 3. Repositorio único: AWS Glue → Amazon RDS (MariaDB)

Una vez que los datos están en sus respectivos almacenamientos intermedios, AWS Glue se encarga de leerlos, normalizarlos y limpiarlos para producir un único dataset consolidado. Glue lee tanto los archivos de S3 (Kaggle y Hugging Face) como los documentos de MongoDB (NewsAPI), aplica la limpieza de texto y unifica todo bajo un esquema común con las columnas `source`, `title`, `content_clean`, `published_at` y `label`.

El resultado se persiste en una instancia de Amazon RDS con motor MariaDB, que actúa como el repositorio único y estructurado del proyecto. La elección de una base de datos relacional para esta fase final responde a que los datos ya limpios y normalizados se prestan perfectamente a consultas SQL, lo que facilita enormemente el análisis exploratorio del Hito 3. La columna `source` permite en todo momento distinguir el origen de cada registro, manteniendo la trazabilidad de los datos sin necesidad de tablas separadas.

## Resumen del flujo

```
[Kaggle CSV]          ──► S3 (raw/kaggle/)      ──┐
                                                   ├──► AWS Glue (ETL) ──► RDS MariaDB
[HuggingFace Parquet] ──► S3 (raw/huggingface/) ──┤                       (raw_news_combined)
                                                   │
[NewsAPI]  ──► Kafka ──► MongoDB Atlas ────────────┘
```